# Symplectic Integration of Planetary Orbits

Each planet in our solar system traces a closed ellipse around the Sun, returning
to the same point year after year. That endless repetition is only possible
because gravity is a *conservative* force: a planet neither gains nor loses total
energy as it orbits. A numerical integrator that quietly leaks energy will make
the simulated orbit slowly spiral inward or drift outward, even though the real
planet does neither.

This notebook models the **two-body heliocentric problem**: each planet is
treated independently as a point mass orbiting a fixed Sun, pulled by Newton's
inverse-square law of gravity. We integrate the same seven planets (Venus through
Neptune) with three integrators of increasing quality:

1. **Euler-Cromer** - a 1st-order *symplectic* method. Already good enough to keep
   the orbits closed for centuries, the same trick used in Session 18's
   `cromer_pendulum.ipynb`.
2. **Velocity Verlet** - a 2nd-order symplectic method that hugs the true energy
   surface far more tightly.
3. **Yoshida 4th-order** - three Verlet substeps composed to cancel lower-order
   error, giving the best long-term fidelity of the three.

For every method we verify **Kepler's Third Law**, $T^2 \propto a^3$, by measuring
each planet's orbital period $T$ directly from the simulated trajectory and
comparing it to its semi-major axis $a$.

Everything the notebook needs is defined locally in the first cell, so it runs
stand-alone with no imports from course utility modules.

---
## Setup: units, gravity, and the planet data

### Astronomical units

Working in SI (meters, kilograms, seconds) would force us to carry enormous
numbers around, so we switch to **astronomical units**:

- distances in **AU** (1 AU = mean Earth-Sun distance $= 1.4960 \times 10^{11}$ m),
- time in **Julian years** (1 yr $= 365.25$ days $= 3.15576 \times 10^7$ s).

In these units the Sun's gravitational parameter is
$\mu = G M_\odot \approx 4\pi^2\ \text{AU}^3/\text{yr}^2$. The convenient
consequence is that Kepler's constant $T^2/a^3$ comes out very close to **1**
(in $\text{yr}^2/\text{AU}^3$) for every planet - a built-in sanity check.

### Newtonian gravity

Each planet obeys the inverse-square law. With the Sun fixed at the origin and
the planet at position $\mathbf{r} = (x, y)$ a distance $r = |\mathbf{r}|$ away,
the acceleration is

$$\mathbf{a} = -\frac{\mu\,\mathbf{r}}{r^3}
  = -\frac{\mu}{r^2}\,\hat{\mathbf{r}}.$$

This single vector equation drives all three integrators; only the *update rule*
that advances position and velocity changes from one method to the next.

The next cell defines the constants, the planet table (name, plot color,
semi-major axis $a$, and reference eccentricity $e$), and the shared
`accel()` helper.

In [ ]:
"""planets.ipynb"""

# Cell 01 - Imports, physical constants, planet data, and shared helpers

%matplotlib inline

from math import ceil, tau

import matplotlib.pyplot as plt
import numpy as np

# Physical constants (SI -> AU/yr)
m_Sun = 1.98847e30  # mass of the Sun (kg)
G = 6.67430e-11  # gravitational constant (m^3 kg^-1 s^-2)
year_in_seconds = 3.15576e7  # one Julian year (s)
AU = 1.495978707e11  # mean Earth-Sun distance (m)

# Sun's gravitational parameter mu = G * M_sun, converted to AU^3 / yr^2
mu = G * m_Sun * (year_in_seconds**2) / (AU**3)

# Planet data: (Name, plot color, semi-major axis a [AU], eccentricity e [-])
# The e values are standard J2000-era reference values (NASA Planet Compare table).
planets = [
    ("Venus", "peru", 0.723332, 0.00677672),
    ("Earth", "dodgerblue", 1.000000, 0.01671123),
    ("Mars", "red", 1.523679, 0.09339410),
    ("Jupiter", "chocolate", 5.2044, 0.04838624),
    ("Saturn", "gold", 9.5826, 0.05386179),
    ("Uranus", "mediumturquoise", 19.2184, 0.04725744),
    ("Neptune", "royalblue", 30.1104, 0.00859048),
]
planet_count = len(planets)

a_au = np.array([p[2] for p in planets], dtype=float)  # semi-major axes (AU)
e_ref = np.array([p[3] for p in planets], dtype=float)  # reference eccentricities


def accel(xv: np.ndarray, yv: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Acceleration components under Newtonian gravity: a = -mu * r / r^3.

    Works elementwise on arrays so all planets are advanced at once.
    """
    r = np.sqrt(xv * xv + yv * yv)
    inv_r3 = 1.0 / (r * r * r)
    return -mu * xv * inv_r3, -mu * yv * inv_r3


# Quick check: print mu (expect ~4*pi^2 ~ 39.48) and the planet table
print(f"Sun's gravitational parameter  mu = {mu:.4f} AU^3/yr^2")
print(f"(expected 4*pi^2 = {4 * np.pi**2:.4f})\n")
print(f"{'Planet':>9}  {'a (AU)':>8}  {'e':>10}")
print("-" * 32)
for name, _, a, e in planets:
    print(f"{name:>9}  {a:>8.4f}  {e:>10.7f}")

---
# Part 1 - Euler-Cromer (1st-order symplectic)

The Euler-Cromer method is the simplest integrator that respects energy
conservation. It differs from plain Forward Euler by a single, crucial detail:
after updating the velocity, it advances the position using the **new** velocity
rather than the old one.

$$
\mathbf{v}_{i+1} = \mathbf{v}_i + \mathbf{a}_i\,\Delta t, \qquad
\mathbf{r}_{i+1} = \mathbf{r}_i + \mathbf{v}_{i+1}\,\Delta t.
$$

That tiny reordering is what makes the method *symplectic*: instead of steadily
injecting energy (Forward Euler spirals a planet outward until it escapes),
Euler-Cromer keeps the total energy oscillating within a small bounded band. The
orbit stays closed for centuries.

### Initial conditions - a circular start

For this first pass we launch every planet on a **circular** orbit of radius $a$:
place it on the $+x$ axis and give it the exact circular speed
$v = \sqrt{\mu / a}$ perpendicular to the radius. Circular orbits are the
cleanest way to see that the integrator keeps a planet at a steady distance.

### Measuring the period

To confirm Kepler's Third Law we need each planet's orbital period $T$. We find it
in a coordinate-independent way: detect successive **outward crossings** of the
radius $r = a$ and take the time between two consecutive crossings. Then we form
the ratio $T^2 / a^3$, which should be $\approx 1$ for every planet.

We run the simulation for **400 Julian years** so even Neptune (period $\approx
165$ yr) completes more than two full orbits. We take **2 steps per day**
($\Delta t = 12$ hours), which is plenty for a stable, closed orbit.

In [ ]:
# Cell 02 - Euler-Cromer integration and Kepler's Third Law

# Simulation settings
tf = 400.0  # final time (Julian years)
steps_per_day = 2  # 2 => dt = 12 hours
n = ceil(tf * 365.25 * steps_per_day)
dt = tf / n  # time step (years)

# Time and state arrays
t = np.arange(n, dtype=float) * dt  # years
x = np.zeros((n, planet_count), dtype=float)  # AU
y = np.zeros((n, planet_count), dtype=float)  # AU
vx = np.zeros((n, planet_count), dtype=float)  # AU/yr
vy = np.zeros((n, planet_count), dtype=float)  # AU/yr

# Initial conditions: circular orbit, planet on +x axis moving in +y direction
x[0, :] = a_au
y[0, :] = 0.0
vx[0, :] = 0.0
vy[0, :] = np.sqrt(mu / a_au)  # circular-orbit speed

# Euler-Cromer update: advance velocity first, then position with the NEW velocity
for i in range(1, n):
    ax, ay = accel(x[i - 1, :], y[i - 1, :])
    vx[i, :] = vx[i - 1, :] + ax * dt
    vy[i, :] = vy[i - 1, :] + ay * dt
    x[i, :] = x[i - 1, :] + vx[i, :] * dt  # uses the freshly updated velocity
    y[i, :] = y[i - 1, :] + vy[i, :] * dt

# Save the trajectory for the plot cell that follows
x_cromer, y_cromer = x.copy(), y.copy()

# Demonstrate Kepler's Third Law: T^2 proportional to a^3
print("Planet Orbits - Kepler's Third Law - Euler-Cromer Integrator:\n")
for j in range(planet_count):
    name, _, a, _ = planets[j]
    # Detect successive outward crossings of r ~ a to estimate one full period
    r = np.sqrt(x[:, j] ** 2 + y[:, j] ** 2)
    crossing_indices = np.where((r[:-1] < a) & (r[1:] >= a))[0]
    T = t[crossing_indices[1]] - t[crossing_indices[0]]  # period (years)
    ratio = T**2 / a**3  # should be ~1 yr^2/AU^3 for every planet
    print(
        f"{name:>9}: T= {T:>8.4f} yr, a= {a:>7.4f} AU, T^2/a^3= {ratio:.6f} yr^2/AU^3"
    )

### The Euler-Cromer orbits

Below are the seven simulated orbits on a black background (the Sun is the yellow
dot at the origin). Because we launched each planet on a circular orbit, the
paths are near-perfect circles. The key result is that after **400 years** the
orbits are still crisp closed curves - a non-symplectic method like Forward Euler
would have smeared them into outward spirals by now.

In [ ]:
# Cell 03 - Euler-Cromer orbit plot

plt.figure("cromer", figsize=(8, 8))
plt.gca().set_facecolor("black")
for j in range(planet_count):
    name, color, _, _ = planets[j]
    plt.plot(x_cromer[:, j], y_cromer[:, j], c=color, label=name)
plt.scatter([0.0], [0.0], s=20, marker="o", c="yellow", label="Sun")
plt.title(f"Planet Orbits - Euler-Cromer ({tf:g} Julian Earth Years)")
plt.xlabel("x (AU)")
plt.ylabel("y (AU)")
plt.gca().set_aspect("equal", adjustable="box")
plt.legend(framealpha=1.0)
plt.tight_layout()
plt.show()

---
# Part 2 - Velocity Verlet (2nd-order symplectic)

Euler-Cromer keeps the energy bounded, but its error is still first order in
$\Delta t$. **Velocity Verlet** doubles the order of accuracy while remaining
symplectic. Each step advances the position using the current acceleration, then
corrects the velocity with the **average** of the accelerations at the start and
end of the step:

$$
\mathbf{r}_{i+1} = \mathbf{r}_i + \mathbf{v}_i\,\Delta t
  + \tfrac12 \mathbf{a}_i\,\Delta t^2, \qquad
\mathbf{v}_{i+1} = \mathbf{v}_i
  + \tfrac12\bigl(\mathbf{a}_i + \mathbf{a}_{i+1}\bigr)\,\Delta t.
$$

That averaging is what makes it 2nd order, and it keeps the long-term energy
error dramatically smaller than Euler-Cromer at the same step size.

### Realistic elliptical orbits

Now that we have a more faithful integrator, we launch each planet on its **real
elliptical orbit** instead of a circle. We start every planet at **perihelion**
(closest approach) on the $+x$ axis, where

$$
r_p = a(1 - e), \qquad v_p = \sqrt{\frac{\mu(1 + e)}{a(1 - e)}}
$$

(the second expression is the **vis-viva** speed at perihelion). Venus's orbit is
nearly circular ($e \approx 0.007$) while Mars is visibly eccentric
($e \approx 0.093$).

### Measuring period and eccentricity from the trajectory

For these elliptical orbits we use two robust estimators:

- **Period** via the **unwrapped true anomaly**. We track the polar angle
  $\theta = \operatorname{atan2}(y, x)$, unwrap it so it grows smoothly past
  $2\pi$, and find the time at which it has advanced by exactly one full turn
  ($\tau = 2\pi$). This is far steadier than watching the radius for
  near-circular orbits.
- **Eccentricity** via the **Laplace-Runge-Lenz (eccentricity) vector**

  $$
  \mathbf{e} = \frac{\mathbf{v} \times \mathbf{h}}{\mu} - \hat{\mathbf{r}},
  $$

  where $\mathbf{h} = \mathbf{r} \times \mathbf{v}$ is the specific angular
  momentum. Its magnitude $|\mathbf{e}|$ equals the orbital eccentricity and,
  for a perfectly conserved orbit, is constant in time. This is much less noisy
  than estimating $e$ from $r_{\min}$ and $r_{\max}$, especially for tiny $e$.

We also report the maximum relative drift in the specific orbital energy
$\varepsilon = \tfrac12 v^2 - \mu/r$, i.e. $\max |\Delta\varepsilon /
\varepsilon_0|$ - a direct measure of how well the integrator conserves energy.

In [ ]:
# Cell 04 - Velocity Verlet integration, Kepler's Law, eccentricity, energy drift


def e_vector(
    xv: np.ndarray, yv: np.ndarray, vxv: np.ndarray, vyv: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Laplace-Runge-Lenz eccentricity vector (and magnitude) for 2D motion."""
    r = np.sqrt(xv * xv + yv * yv)
    h = xv * vyv - yv * vxv  # specific angular momentum (z-component)
    ex = (vyv * h) / mu - (xv / r)
    ey = (-vxv * h) / mu - (yv / r)
    return ex, ey, np.sqrt(ex * ex + ey * ey)


# Simulation settings (same span, 2 steps/day keeps the notebook fast)
tf = 400.0  # final time (Julian years)
steps_per_day = 2  # 2 => dt = 12 hours
n = ceil(tf * 365.25 * steps_per_day)
dt = tf / n

t = np.arange(n, dtype=float) * dt
x = np.zeros((n, planet_count), dtype=float)
y = np.zeros((n, planet_count), dtype=float)
vx = np.zeros((n, planet_count), dtype=float)
vy = np.zeros((n, planet_count), dtype=float)

# Initial conditions: start at perihelion on +x axis with vis-viva speed
r_p = a_au * (1.0 - e_ref)
v_p = np.sqrt(mu * (1.0 + e_ref) / (a_au * (1.0 - e_ref)))
x[0, :] = r_p
y[0, :] = 0.0
vx[0, :] = 0.0
vy[0, :] = v_p

# Velocity Verlet integration
ax0, ay0 = accel(x[0, :], y[0, :])
for i in range(1, n):
    # Drift: advance position using current velocity and acceleration
    x[i, :] = x[i - 1, :] + vx[i - 1, :] * dt + 0.5 * ax0 * dt * dt
    y[i, :] = y[i - 1, :] + vy[i - 1, :] * dt + 0.5 * ay0 * dt * dt
    # New acceleration at the updated position
    ax1, ay1 = accel(x[i, :], y[i, :])
    # Kick: advance velocity using the average of old and new acceleration
    vx[i, :] = vx[i - 1, :] + 0.5 * (ax0 + ax1) * dt
    vy[i, :] = vy[i - 1, :] + 0.5 * (ay0 + ay1) * dt
    ax0, ay0 = ax1, ay1

x_verlet, y_verlet = x.copy(), y.copy()

print("Planet Orbits - Kepler's Third Law - Velocity Verlet Integrator:\n")
for j in range(planet_count):
    name, _, a, e0 = planets[j]
    # Period from the unwrapped true anomaly advancing by one full turn (tau)
    th = np.unwrap(np.arctan2(y[:, j], x[:, j]))
    k = np.searchsorted(th, th[0] + tau)
    if k <= 1 or k >= len(th):
        print(f"{name:>9}: insufficient span to estimate period/eccentricity")
        continue
    T = t[k] - t[0]
    ratio = T**2 / a**3
    # Eccentricity from the mean magnitude of the LRL vector over one orbit
    _, _, e_series = e_vector(
        x[: k + 1, j], y[: k + 1, j], vx[: k + 1, j], vy[: k + 1, j]
    )
    ehat = float(np.mean(e_series))
    pct = (ehat - e0) / e0 * 100.0 if e0 != 0 else np.nan
    # Maximum relative drift in specific orbital energy over the whole run
    r = np.sqrt(x[:, j] ** 2 + y[:, j] ** 2)
    v2 = vx[:, j] ** 2 + vy[:, j] ** 2
    eps = 0.5 * v2 - mu / r
    eps_drift = np.max(np.abs((eps - eps[0]) / eps[0]))
    print(
        f"{name:>9}: T= {T:>8.4f} yr, a= {a:>7.4f} AU,"
        f" T^2/a^3= {ratio:.6f}"
        f"  e= {ehat:>9.7f} ({pct:+7.4f}%)"
        f"  max |ΔE/E|= {eps_drift:.2e}"
    )

### The Velocity Verlet orbits

With the elliptical initial conditions the paths are now true ellipses with the
Sun at one focus. The eccentricities are small enough that the orbits still look
nearly circular at this scale, but the printed table above confirms Verlet
recovers each planet's reference $e$ to a fraction of a percent, and the energy
drift $|\Delta E / E|$ is tiny - orders of magnitude smaller than a
non-symplectic method would give over the same 400-year run.

In [ ]:
# Cell 05 - Velocity Verlet orbit plot

plt.figure("verlet", figsize=(8, 8))
plt.gca().set_facecolor("black")
for j in range(planet_count):
    name, color, _, _ = planets[j]
    plt.plot(x_verlet[:, j], y_verlet[:, j], c=color, label=name)
plt.scatter([0.0], [0.0], s=20, marker="o", c="yellow", label="Sun")
plt.title(f"Planet Orbits - Velocity Verlet ({tf:g} Julian Earth Years)")
plt.xlabel("x (AU)")
plt.ylabel("y (AU)")
plt.gca().set_aspect("equal", adjustable="box")
plt.legend(loc="upper right", framealpha=1.0)
plt.tight_layout()
plt.show()

---
# Part 3 - Yoshida 4th-order (the cleanest)

Yoshida's method (Yoshida, 1990, *Construction of higher order symplectic
integrators*) pushes accuracy to **4th order** while staying symplectic. The idea
is elegant: take three Velocity-Verlet substeps of carefully chosen lengths and
compose them so their leading error terms cancel. The middle substep even steps
**backward** in time (its weight $w_0$ is negative) to erase the 2nd-order error
left by the outer two.

The velocity weights are

$$
w_1 = \frac{1}{2 - 2^{1/3}}, \qquad w_0 = \frac{-2^{1/3}}{2 - 2^{1/3}},
$$

applied in the order $(w_1, w_0, w_1)$ with each substep taking $\text{weight}
\times \Delta t$ of simulated time. Note $w_1 + w_0 + w_1 = 1$, so one full step
still advances time by exactly $\Delta t$.

We inline the coefficient calculation directly (the original script imported it
from a course utility module, but this notebook stays self-contained). Everything
else - the perihelion initial conditions, the unwrapped-anomaly period, and the
LRL eccentricity - is identical to the Verlet section, so we can compare the two
methods on equal footing.

In [ ]:
# Cell 06 - Yoshida 4th-order integration, Kepler's Law, eccentricity, energy drift


def yoshida_coeffs() -> tuple[np.ndarray, np.ndarray]:
    """Position (c) and velocity (d) substep weights for Yoshida 4th order."""
    cbrt2 = 2.0 ** (1.0 / 3.0)
    w1 = 1.0 / (2.0 - cbrt2)
    w0 = -cbrt2 / (2.0 - cbrt2)
    c = np.array([w1 / 2.0, (w0 + w1) / 2.0, (w0 + w1) / 2.0, w1 / 2.0])
    d = np.array([w1, w0, w1])
    return c, d


# Only the velocity (d) weights are needed; vv_substep handles the half-drifts.
_, _d = yoshida_coeffs()
y4_coeffs = tuple(_d)  # (w1, w0, w1)


def vv_substep(
    x0: np.ndarray, y0: np.ndarray, vx0: np.ndarray, vy0: np.ndarray, h: float
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """One Velocity-Verlet (kick-drift-kick) substep of duration h (years)."""
    ax0, ay0 = accel(x0, y0)
    vxh = vx0 + 0.5 * h * ax0  # half kick
    vyh = vy0 + 0.5 * h * ay0
    x1 = x0 + h * vxh  # full drift
    y1 = y0 + h * vyh
    ax1, ay1 = accel(x1, y1)
    vx1 = vxh + 0.5 * h * ax1  # half kick
    vy1 = vyh + 0.5 * h * ay1
    return x1, y1, vx1, vy1


# Simulation settings
tf = 400.0
steps_per_day = 2  # 2 => dt = 12 hours
n = ceil(tf * 365.25 * steps_per_day)
dt = tf / n

t = np.arange(n, dtype=float) * dt
x = np.zeros((n, planet_count), dtype=float)
y = np.zeros((n, planet_count), dtype=float)
vx = np.zeros((n, planet_count), dtype=float)
vy = np.zeros((n, planet_count), dtype=float)

# Same perihelion initial conditions as the Verlet section
r_p = a_au * (1.0 - e_ref)
v_p = np.sqrt(mu * (1.0 + e_ref) / (a_au * (1.0 - e_ref)))
x[0, :] = r_p
y[0, :] = 0.0
vx[0, :] = 0.0
vy[0, :] = v_p

# Yoshida step = three weighted Velocity-Verlet substeps
for i in range(1, n):
    xk, yk = x[i - 1, :].copy(), y[i - 1, :].copy()
    vxk, vyk = vx[i - 1, :].copy(), vy[i - 1, :].copy()
    for c in y4_coeffs:
        xk, yk, vxk, vyk = vv_substep(xk, yk, vxk, vyk, c * dt)
    x[i, :], y[i, :] = xk, yk
    vx[i, :], vy[i, :] = vxk, vyk

x_yoshida, y_yoshida = x.copy(), y.copy()

print("Planet Orbits - Kepler's Third Law - Yoshida 4th-Order Integrator:\n")
for j in range(planet_count):
    name, _, a, e0 = planets[j]
    th = np.unwrap(np.arctan2(y[:, j], x[:, j]))
    k = np.searchsorted(th, th[0] + tau)
    if k <= 1 or k >= len(th):
        print(f"{name:>9}: insufficient span to estimate period/eccentricity")
        continue
    T = t[k] - t[0]
    ratio = T**2 / a**3
    _, _, e_series = e_vector(
        x[: k + 1, j], y[: k + 1, j], vx[: k + 1, j], vy[: k + 1, j]
    )
    ehat = float(np.mean(e_series))
    pct = (ehat - e0) / e0 * 100.0 if e0 != 0 else np.nan
    r = np.sqrt(x[:, j] ** 2 + y[:, j] ** 2)
    v2 = vx[:, j] ** 2 + vy[:, j] ** 2
    eps = 0.5 * v2 - mu / r
    eps_drift = np.max(np.abs((eps - eps[0]) / eps[0]))
    print(
        f"{name:>9}: T= {T:>8.4f} yr, a= {a:>7.4f} AU,"
        f" T^2/a^3= {ratio:.6f}"
        f"  e= {ehat:>9.7f} ({pct:+7.4f}%)"
        f"  max |ΔE/E|= {eps_drift:.2e}"
    )

### The Yoshida orbits

The plotted orbits look the same as Verlet's to the eye - both are visually
perfect ellipses. The difference lives in the numbers: compare the
$|\Delta E / E|$ column of this table with the Verlet one. Yoshida's 4th-order
composition typically drives the energy drift down by several more orders of
magnitude, which is exactly what you want for very long integrations (millions of
years of solar-system evolution).

**Takeaway:** for a conservative system like an orbit, being *symplectic* matters
more than raw per-step accuracy. Euler-Cromer already keeps the orbits closed;
Velocity Verlet tightens the energy conservation by orders of magnitude; and
Yoshida's 4th-order method tightens it further still - all three recover Kepler's
Third Law with $T^2/a^3 \approx 1$ for every planet.

In [ ]:
# Cell 07 - Yoshida 4th-order orbit plot

plt.figure("yoshida", figsize=(8, 8))
plt.gca().set_facecolor("black")
for j in range(planet_count):
    name, color, _, _ = planets[j]
    plt.plot(x_yoshida[:, j], y_yoshida[:, j], c=color, label=name)
plt.scatter([0.0], [0.0], s=20, marker="o", c="yellow", label="Sun")
plt.title(f"Planet Orbits - Yoshida 4th-Order ({tf:g} Julian Earth Years)")
plt.xlabel("x (AU)")
plt.ylabel("y (AU)")
plt.gca().set_aspect("equal", adjustable="box")
plt.legend(loc="upper right", framealpha=1.0)
plt.tight_layout()
plt.show()